In [1]:
#imports 
import rasterio
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, accuracy_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier

## Importing data:

In [2]:
back = rasterio.open("/kaggle/input/competitions/geohab-mlwg-competition-2026/MBES/backscatter.tif")
bath = rasterio.open("/kaggle/input/competitions/geohab-mlwg-competition-2026/MBES/bathymetry.tif")
train_df = pd.read_csv('/kaggle/input/competitions/geohab-mlwg-competition-2026/train.csv')
test_df = pd.read_csv('/kaggle/input/competitions/geohab-mlwg-competition-2026/test.csv')

In [3]:
print('Train set: ', train_df.shape)
display(train_df.head())

print('Test set: ', test_df.shape)
display(test_df.head())

back_data = back.read(1)
print('Backscatter data: ',back_data.shape)
print(back_data[:5, :5])

bath_data = bath.read(1)
print('Bathymestry data: ', bath_data.shape)
print(bath_data[:5, :5])

Train set:  (6256, 3)


,class,x,y
0,NVB,453594.477237,5.679192e+06
1,FMAT,453561.906453,5.679109e+06
2,ALG,453744.452238,5.679033e+06
3,ALG,453863.445302,5.679038e+06
4,ALG,453964.611906,5.679017e+06


Test set:  (98, 3)


,ID,x,y
0,1,453702.166779,5.679044e+06
1,2,454126.252800,5.678999e+06
2,3,453957.881092,5.678942e+06
3,4,453798.917484,5.678955e+06
4,5,453520.953671,5.679124e+06


Backscatter data:  (4040, 4743)
[[-10000. -10000. -10000. -10000. -10000.]
 [-10000. -10000. -10000. -10000. -10000.]
 [-10000. -10000. -10000. -10000. -10000.]
 [-10000. -10000. -10000. -10000. -10000.]
 [-10000. -10000. -10000. -10000. -10000.]]
Bathymestry data:  (4040, 4743)
[[-10000. -10000. -10000. -10000. -10000.]
 [-10000. -10000. -10000. -10000. -10000.]
 [-10000. -10000. -10000. -10000. -10000.]
 [-10000. -10000. -10000. -10000. -10000.]
 [-10000. -10000. -10000. -10000. -10000.]]


## Extracting features:

In [4]:
def get_features(x, y):
    row, col = bath.index(x, y)
    return bath_data[row, col], back_data[row, col]

train_df[['depth', 'scatter']] = train_df.apply(
    lambda r: pd.Series(get_features(r['x'], r['y'])),
    axis=1
)

test_df[['depth', 'scatter']] = test_df.apply(
    lambda r: pd.Series(get_features(r['x'], r['y'])),
    axis=1
)

In [5]:
train_df.info()
test_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6256 entries, 0 to 6255
Data columns (total 5 columns):
 #   Column   Non-Null Count  Dtype  
---  ------   --------------  -----  
 0   class    6256 non-null   object 
 1   x        6256 non-null   float64
 2   y        6256 non-null   float64
 3   depth    6256 non-null   float32
 4   scatter  6256 non-null   float32
dtypes: float32(2), float64(2), object(1)
memory usage: 195.6+ KB
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 98 entries, 0 to 97
Data columns (total 5 columns):
 #   Column   Non-Null Count  Dtype  
---  ------   --------------  -----  
 0   ID       98 non-null     int64  
 1   x        98 non-null     float64
 2   y        98 non-null     float64
 3   depth    98 non-null     float32
 4   scatter  98 non-null     float32
dtypes: float32(2), float64(2), int64(1)
memory usage: 3.2 KB


In [6]:
display(train_df[['depth','scatter']].describe())
display(test_df[['depth','scatter']].describe())

,depth,scatter
count,6256.000000,6256.000000
mean,-8.568404,-23.904961
std,2.939247,4.177290
min,-20.247799,-39.139683
25%,-10.757194,-26.859035
50%,-8.688892,-24.019592
75%,-6.291407,-21.499985
max,-1.119716,-12.049013


,depth,scatter
count,98.000000,98.000000
mean,-9.540840,-24.256353
std,4.336661,4.373584
min,-21.843966,-35.359051
25%,-11.946906,-27.488937
50%,-9.491405,-24.019592
75%,-6.662931,-21.819818
max,-2.299802,-14.259773


It can be observerd that the **min** values of test data are insider the bounds of the training set but that isnt the case for the **max** values, so the model is bound to cause some prediction errors if it heavily relies only on these two features

## Baseline model

We will be training a **LogisticRegression** model since a **RandomForest** and a **decisiontree** would be a overkill on 2 features 

In [7]:
target = 'class'

X = train_df[["depth","scatter"]]
y = train_df[target].map({'SGAM': 0, 'NVB': 1, 'SGZ': 2, 'ALG': 3, 'FMAT': 4})

X_test = test_df[['depth','scatter']]

In [8]:
#splitting the data into train/val splits
X_train , X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, stratify=y, random_state=24)

### LogisticRegression

In [9]:
#Model
model = LogisticRegression(
    multi_class='multinomial',
    solver='lbfgs'
)

pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('model', model)
])

pipeline.fit(X_train, y_train)

/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(


Pipeline(steps=[('scaler', StandardScaler()),
                ('model', LogisticRegression(multi_class='multinomial'))])

In [10]:
val_preds = pipeline.predict(X_val)
val_acc = accuracy_score(y_val, val_preds)
val_f1 = f1_score(y_val, val_preds, average='weighted')

print('Baseline validation accuracy: ', val_acc)
print('Baseline validation f1: ', val_f1)

Baseline validation accuracy:  0.5638977635782748
Baseline validation f1:  0.5066511832034243


### RandomForestCLassifier
Even though our data is limited a tree based model can find good splits that can work in favor for classification tasks 

In [11]:
model = RandomForestClassifier(random_state=42)

model.fit(X_train, y_train)

RandomForestClassifier(random_state=42)

In [12]:
val_preds = model.predict(X_val)
val_acc = accuracy_score(y_val, val_preds)
val_f1 = f1_score(y_val, val_preds, average='weighted')

print('Baseline validation accuracy: ', val_acc)
print('Baseline validation f1: ', val_f1)

Baseline validation accuracy:  0.6645367412140575
Baseline validation f1:  0.6621163675064173


as can be observed a tree based model worked better than logistic regression

## Final predictions 

In [13]:
y_preds = model.predict(X_test)

submission = pd.DataFrame({
    'ID': test_df['ID'],
    target: y_preds
})

submission[target] = submission[target].map({0: 'SGAM', 1: 'NVB', 2: 'SGZ', 3: 'ALG', 4: 'FMAT'})

submission.to_csv('submission.csv', index=False)